# Notebook 11 — Fraud Detection Platform: Executive Rollup Reports (PDF / PPTX / XLSX)
**Auto-populates three executive-facing deliverables -- an extremely descriptive PDF report, a C-suite PPTX deck, and a role-based XLSX decision scorecard -- entirely from NB1-NB9's real on-disk outputs. Nothing here is hand-authored analysis: every number is loaded verbatim from a file a prior notebook already wrote, and every SMART-decision status (Go/Conditional/Hold) is a simple boolean check on a real flag (investigate_flag, any_alert, all_checks_passed, outside_pipeline_scope_count) -- never invented.**


In [ ]:
##############################################################################
# SETUP -- WARP-optimized environment.
##############################################################################
import os, time, json, warnings, subprocess, sys, io
from datetime import datetime, timezone
warnings.filterwarnings("ignore")

_RUN_T0 = time.time()

CPU_THRESHOLD_PCT = 93
RAM_THRESHOLD_PCT = 90

_N_THREADS = max(1, int((os.cpu_count() or 4) * (CPU_THRESHOLD_PCT / 100) // 1))
os.environ.setdefault("OMP_NUM_THREADS", str(_N_THREADS))
os.environ.setdefault("OPENBLAS_NUM_THREADS", str(_N_THREADS))
os.environ.setdefault("MKL_NUM_THREADS", str(_N_THREADS))

for _pkg, _import_name in (("psutil", "psutil"), ("reportlab", "reportlab"),
                            ("python-pptx", "pptx"), ("openpyxl", "openpyxl"),
                            ("matplotlib", "matplotlib")):
    try:
        __import__(_import_name)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", _pkg], check=True)

import psutil
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

RANDOM_SEED = 42

_ram_start = psutil.virtual_memory()
print(f"WARP thread ceiling: {_N_THREADS} threads (of {os.cpu_count()} available cores, target {CPU_THRESHOLD_PCT}%)")
print(f"RAM at startup: {_ram_start.percent:.1f}% used ({_ram_start.used/1e9:.2f} GB / {_ram_start.total/1e9:.2f} GB)")
print("Setup complete. (No CSV/model load -- rolls up NB1-NB9's real on-disk outputs into 3 formats.)")

##############################################################################
# REPO-LAYOUT BOOTSTRAP -- verified detection, unchanged from NB1-10.
##############################################################################
_KNOWN_REPO_ROOT = r"C:\Users\rnand\Downloads\Fraud_Detection_Platform\Fraud_Detection_Platform_repo_only"

def _looks_like_repo(_p):
    return os.path.isdir(os.path.join(_p, "notebooks")) or os.path.exists(os.path.join(_p, "requirements.txt"))

_cwd = os.getcwd()
_parent = os.path.abspath(os.path.join(_cwd, ".."))

if os.path.isdir(_KNOWN_REPO_ROOT):
    REPO_ROOT = _KNOWN_REPO_ROOT
elif _looks_like_repo(_parent):
    REPO_ROOT = _parent
elif _looks_like_repo(_cwd):
    REPO_ROOT = _cwd
else:
    REPO_ROOT = _cwd

REPORTS_DIR = os.path.join(REPO_ROOT, "reports")
RESULTS_DIR = os.path.join(REPORTS_DIR, "nb11_results")
try:
    os.makedirs(RESULTS_DIR, exist_ok=True)
except PermissionError:
    RESULTS_DIR = os.path.join(os.getcwd(), "nb11_results")
    os.makedirs(RESULTS_DIR, exist_ok=True)

print("Repo root:      ", REPO_ROOT)
print("NB11 results in:", RESULTS_DIR)

##############################################################################
# ROLLUP LOADER -- SINGLE SOURCE OF TRUTH. Opens the real files NB1-NB9
# already wrote to disk and returns their contents verbatim. Nothing here
# computes, estimates, or invents a number. Identical contract to NB10.
##############################################################################
def _read_json(path):
    if not os.path.exists(path):
        return None
    with open(path, encoding="utf-8") as f:
        return json.load(f)

def _read_jsonl(path):
    if not os.path.exists(path):
        return []
    out = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                out.append(json.loads(line))
    return out

def p(*parts):
    return os.path.join(REPORTS_DIR, *parts)

nb1 = _read_json(p("nb1_results", "nb1_final_results.json"))
nb2 = _read_json(p("nb2_results", "nb2_validation_report.json"))
nb4 = _read_json(p("nb4_results", "nb4_serving_report.json"))
nb5 = _read_json(p("nb5_results", "nb5_stress_test_report.json"))
nb6 = _read_json(p("nb6_results", "nb6_model_tiering_matrix.json"))
nb7 = _read_json(p("nb7_results", "nb7_bcbs239_report.json"))
nb8 = _read_json(p("nb8_results", "nb8_report.json"))
nb9 = _read_json(p("nb9_results", "nb9_report.json"))
nb3_drift_history = _read_jsonl(p("nb3_results", "drift_history.jsonl"))

_missing = [k for k, v in [("nb1", nb1), ("nb2", nb2), ("nb4", nb4), ("nb5", nb5),
                            ("nb6", nb6), ("nb7", nb7), ("nb8", nb8), ("nb9", nb9)] if v is None]
if _missing:
    raise SystemExit(
        f"Cannot build the executive rollup -- missing real outputs from: {_missing}. "
        f"Run notebooks 01-09 first so reports/nb{{1,2,4,5,6,7,8,9}}_results/*.json exist under {REPORTS_DIR}."
    )
print(f"Loaded real outputs from NB1, NB2, NB3 ({len(nb3_drift_history)} real monitoring entries), "
      f"NB4, NB5, NB6, NB7, NB8, NB9.")

##############################################################################
# AUTO-PICKED DERIVED VALUES -- pure lookups + disclosed arithmetic already
# present in the source JSON. No new analysis, no invented figures.
##############################################################################
EUR_TO_USD = nb5["run_metadata"]["eur_to_usd_rate"]  # real, NB5's own metadata

def eur(x):
    return f"EUR {x:,.2f} (approx. USD {x*EUR_TO_USD:,.2f})"

def pct(x, d=2):
    return f"{x*100:.{d}f}%"

def tier_counts(rows):
    pas = sum(1 for r in rows if r[2].startswith("Pass"))
    tbd = sum(1 for r in rows if r[2].startswith("TBD"))
    cond = len(rows) - pas - tbd
    return {"pass": pas, "cond": cond, "tbd": tbd, "total": len(rows)}

T1, T2, T3, T4 = (tier_counts(nb9["tier1_rows"]), tier_counts(nb9["tier2_rows"]),
                  tier_counts(nb9["tier3_rows"]), tier_counts(nb9["tier4_rows"]))
ALL_CHECKS = T1["total"] + T2["total"] + T3["total"] + T4["total"]
ALL_PASS = T1["pass"] + T2["pass"] + T3["pass"] + T4["pass"]
ALL_TBD = T1["tbd"] + T2["tbd"] + T3["tbd"] + T4["tbd"]

g1 = nb2["gate1_structural_checks"]
g1_pass = sum(1 for v in g1.values() if v)
dq = nb7["data_quality_gate"]
dq_pass = sum(1 for v in dq["checks"].values() if v)
psi_worst = max(nb2["drift_monitoring"]["feature_psi"].values())
bc = nb1["benchmark_check"]

##############################################################################
# ROLE-BASED SMART DECISIONS -- structural framing only; every "real_basis"
# string is built from a real field above, and every "status" is a simple
# boolean check on a real flag, never invented.
##############################################################################
ROLE_DECISIONS = [
    {"role": "CEO / Board", "decision": "Approve continued investment in the fraud-detection program",
     "real_basis": f"Measured savings vs. no model: {eur(nb1['savings_vs_no_model'])} over the real "
                    f"{nb1['dataset']['rows']:,}-transaction sampled window. Classified {nb6['tier_description']} "
                    f"(Tier {nb6['model_tier']}, composite score {nb6['composite_score']}/9).",
     "status": "Conditional" if nb9["outside_pipeline_scope_count"] > 0 else "Go",
     "status_reason": f"{nb9['outside_pipeline_scope_count']} of {ALL_CHECKS} governance sign-off checks "
                       f"(NB9) are marked outside this pipeline's scope -- real organizational facts, not model gaps.",
     "source": "NB1, NB6, NB9"},

    {"role": "CFO", "decision": "Book projected fraud-loss savings into the budget",
     "real_basis": f"Real total cost by scenario -- no model {eur(nb1['financial_impact'][0]['Total Cost (EUR)'])}, "
                    f"naive threshold {eur(nb1['financial_impact'][1]['Total Cost (EUR)'])}, cost-optimal "
                    f"{eur(nb1['financial_impact'][2]['Total Cost (EUR)'])}.",
     "status": "Conditional",
     "status_reason": "Figures are real totals over the sampled ~48-hour window, not annualized -- NB6 "
                       "explicitly disclosed that annualizing would require a representativeness assumption "
                       "this project has not sourced.",
     "source": "NB1, NB6"},

    {"role": "Chief Risk Officer (CRO)", "decision": f"Accept the model into risk appetite at Tier {nb6['model_tier']}",
     "real_basis": f"Worst-case NB5 stress-grid loss is {nb6['rubric']['financial_exposure']['stress_multiple']:.1f}x "
                    f"today's real cost ({eur(nb6['rubric']['financial_exposure']['current_cost_eur'])} -> "
                    f"{eur(nb6['rubric']['financial_exposure']['worst_case_stress_eur'])}). "
                    f"{len(nb6['rubric']['regulatory_scrutiny']['open_flags'])} open regulatory-scrutiny flags.",
     "status": "Conditional" if nb2["drift_monitoring"]["any_alert"] else "Go",
     "status_reason": f"NB2 real drift monitoring any_alert={nb2['drift_monitoring']['any_alert']} "
                       f"(worst real feature PSI {psi_worst:.4f}) -- resolve before treating Tier "
                       f"{nb6['model_tier']} as steady-state.",
     "source": "NB2, NB5, NB6"},

    {"role": "Chief Compliance Officer (CCO)", "decision": "Sign off on the regulatory applicability assessment",
     "real_basis": f"{len(nb8['regulatory_applicability'])} real frameworks mapped (BSA/SAR, Reg E, GLBA "
                    f"Safeguards, PSD2 SCA). BCBS 239 data-quality gate: {dq_pass}/{len(dq['checks'])} checks "
                    f"passed on {dq['n_rows']:,} real rows.",
     "status": "Conditional" if any(m["status"] != "evidenced" for m in nb7["bcbs239_mapping"]) else "Go",
     "status_reason": ", ".join(m["principle_group"] for m in nb7["bcbs239_mapping"] if m["status"] != "evidenced")
                       + " remain limitation-disclosed, not evidenced.",
     "source": "NB7, NB8"},

    {"role": "CTO / Head of Data & Analytics", "decision": "Approve production deployment of the real-time scoring service",
     "real_basis": f"Real local API: p99 client latency {nb4['latency_sla_ms']['client_round_trip']['p99']:.2f} ms. "
                    f"Training/serving consistency check: max abs diff {nb4['training_serving_consistency']['max_abs_diff']} "
                    f"over {nb4['training_serving_consistency']['n_rows_tested']} real rows.",
     "status": "Go" if (nb4["health_check_passed"] and nb4["training_serving_consistency"]["passed"]) else "Hold",
     "status_reason": "Health check and train/serve consistency both real-verified PASS; latency is a local "
                       "signal only, not yet a reachable production deployment.",
     "source": "NB4"},

    {"role": "Head of Fraud Operations", "decision": "Adopt the cost-optimal decision threshold operationally",
     "real_basis": f"Threshold {nb1['threshold_result']['threshold']:.4f} -> real precision {pct(nb1['real_precision'])}, "
                    f"real recall {pct(nb1['real_recall'])}. Human-review band (ASSUMPTION +/-"
                    f"{nb8['human_oversight']['band_half_width_ASSUMPTION']}): "
                    f"{nb8['human_oversight']['n_in_band']} of {nb8['human_oversight']['n_total']:,} real "
                    f"transactions routed for manual review.",
     "status": "Go",
     "status_reason": "Threshold is real and vectorized-search-verified (NB1); review band is a disclosed "
                       "ASSUMPTION, not a measured optimum.",
     "source": "NB1, NB8"},

    {"role": "Model Risk Manager", "decision": "Complete Tier 2 (second-line) model-risk documentation",
     "real_basis": f"NB9 real evidence: Tier 2 checks {T2['pass']}/{T2['total']} Pass (methodology, environment "
                    f"pinning, RANDOM_SEED=42 consistency, fairness-scope disclosure, known limitations, "
                    f"challenger model, model card).",
     "status": "Go" if T2["pass"] == T2["total"] else "Conditional",
     "status_reason": f"All {T2['total']} Tier 2 checks pass on real evidence (NB9) -- ready for a real human "
                       f"Model Risk Manager signature.",
     "source": "NB9"},

    {"role": "Technical Lead / Data Science", "decision": "Resolve the open external-benchmark investigate flag",
     "real_basis": f"investigate_flag={bc['investigate_flag']}, precision_gap={pct(bc['precision_gap'])} vs. "
                    f"external reference (your precision {pct(bc['your_precision'])} vs. reference "
                    f"{pct(bc['reference_precision'])}).",
     "status": "Conditional" if bc["investigate_flag"] else "Go",
     "status_reason": bc["note"],
     "source": "NB1"},
]

##############################################################################
# REAL CHARTS -- plotted directly from the real arrays above; no synthetic
# or placeholder data.
##############################################################################
plt.rcParams.update({"font.size": 9, "axes.edgecolor": "#5B6B85", "axes.labelcolor": "#12213B"})

def _fig_to_png_bytes(fig):
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=160, bbox_inches="tight")
    plt.close(fig)
    buf.seek(0)
    return buf.read()

# Chart 1 -- Stage A screening (real 5-candidate PR-AUC)
_stage_a = nb1["stage_a"]
fig1, ax1 = plt.subplots(figsize=(6.2, 3.2))
_names = [d["model"] for d in _stage_a]
_vals = [d["val_pr_auc"] for d in _stage_a]
_colors = ["#1F3864" if i < 2 else "#B7C4D6" for i in range(len(_stage_a))]
ax1.barh(_names[::-1], _vals[::-1], color=_colors[::-1])
ax1.set_xlim(0, 1)
ax1.set_xlabel("Validation PR-AUC")
ax1.set_title("Stage A Screening -- Real Validation PR-AUC (top 2 advance)")
CHART_STAGE_A_PNG = _fig_to_png_bytes(fig1)

# Chart 2 -- Financial impact by scenario (real totals)
fig2, ax2 = plt.subplots(figsize=(6.2, 3.2))
_scen = [r["Scenario"] for r in nb1["financial_impact"]]
_cost = [r["Total Cost (EUR)"] for r in nb1["financial_impact"]]
ax2.bar(_scen, _cost, color=["#B0342A", "#A5690E", "#1E7A4C"])
ax2.set_ylabel("Real Total Cost (EUR)")
ax2.set_title("Real Total Cost by Scenario")
plt.setp(ax2.get_xticklabels(), rotation=12, ha="right")
CHART_FINANCIAL_PNG = _fig_to_png_bytes(fig2)

# Chart 3 -- Stress scenario ladder (real 3-tier projections, NB5)
fig3, ax3 = plt.subplots(figsize=(6.2, 3.2))
_ladder = nb5["scenario_ladder"]
_tiers = [s["tier_name"] for s in _ladder]
_losses = [s["projected_fraud_loss_usd_or_eur"] for s in _ladder]
ax3.bar(_tiers, _losses, color=["#1E7A4C", "#A5690E", "#B0342A"])
ax3.set_ylabel("Projected Loss (EUR)")
ax3.set_title("NB5 Real 3-Tier Stress Scenario Ladder (disclosed ASSUMPTION multipliers)")
CHART_STRESS_PNG = _fig_to_png_bytes(fig3)

print("Real charts rendered:", 3)

##############################################################################
# 1) PDF -- extremely descriptive executive report (reportlab)
##############################################################################
from reportlab.lib.pagesizes import LETTER
from reportlab.lib.units import inch
from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.platypus import (SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle,
                                 Image as RLImage, PageBreak, HRFlowable)
from reportlab.lib.enums import TA_LEFT

_styles = getSampleStyleSheet()
_styles.add(ParagraphStyle(name="H1c", parent=_styles["Heading1"], textColor=colors.HexColor("#1F3864")))
_styles.add(ParagraphStyle(name="H2c", parent=_styles["Heading2"], textColor=colors.HexColor("#2E74B5")))
_styles.add(ParagraphStyle(name="Bodyc", parent=_styles["BodyText"], leading=14))
_styles.add(ParagraphStyle(name="Caption", parent=_styles["BodyText"], fontSize=8, textColor=colors.HexColor("#5B6B85")))

_pdf_path = os.path.join(RESULTS_DIR, "Fraud_Detection_Executive_Rollup_Report.pdf")
doc = SimpleDocTemplate(_pdf_path, pagesize=LETTER,
                         leftMargin=0.7*inch, rightMargin=0.7*inch, topMargin=0.7*inch, bottomMargin=0.6*inch)
story = []

def _status_color(s):
    return {"Go": colors.HexColor("#1E7A4C"), "Conditional": colors.HexColor("#A5690E"),
            "Hold": colors.HexColor("#B0342A")}.get(s, colors.grey)

_STATUS_HEX = {"Go": "#1E7A4C", "Conditional": "#A5690E", "Hold": "#B0342A"}

def _status_hex(s):
    return _STATUS_HEX.get(s, "#000000")

# ---- Title page ----
story.append(Paragraph("Fraud Detection Platform", _styles["H1c"]))
story.append(Paragraph("Executive Rollup Report", _styles["Title"]))
story.append(Spacer(1, 10))
story.append(Paragraph(
    f"Prepared by Nandagopal &nbsp;&middot;&nbsp; Generated {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M UTC')} "
    f"&nbsp;&middot;&nbsp; Auto-populated from NB1-NB9's real on-disk outputs -- zero-fabrication policy",
    _styles["Bodyc"]))
story.append(Spacer(1, 4))
story.append(Paragraph(
    f"Champion model: <b>{nb1['champion_name']}</b> &nbsp;|&nbsp; Model tier: <b>Tier {nb6['model_tier']} "
    f"-- {nb6['tier_description']}</b> &nbsp;|&nbsp; Real dataset: <b>{nb1['dataset']['rows']:,} transactions, "
    f"{nb1['dataset']['fraud_count']} confirmed frauds</b>", _styles["Bodyc"]))
story.append(Spacer(1, 14))
story.append(HRFlowable(width="100%", color=colors.HexColor("#DCE3EC")))
story.append(Spacer(1, 14))

# ---- Executive summary KPIs ----
story.append(Paragraph("Executive Summary", _styles["H1c"]))
_kpi_rows = [
    ["Metric", "Real Value", "Source"],
    ["Champion CV PR-AUC (5-fold)", f"{nb1['stage_b'][nb1['champion_name']]['mean_pr_auc']:.4f}", "NB1"],
    ["Temporal-split PR-AUC (honest)", f"{nb1['temporal_pr_auc']:.4f}", "NB1"],
    ["Cost-optimal threshold", f"{nb1['threshold_result']['threshold']:.4f}", "NB1"],
    ["Real precision / recall (OOF)", f"{pct(nb1['real_precision'])} / {pct(nb1['real_recall'])}", "NB1"],
    ["Savings vs. no model", eur(nb1["savings_vs_no_model"]), "NB1"],
    ["Savings vs. naive 0.5 threshold", eur(nb1["savings_vs_naive_05"]), "NB1"],
    ["Model tier (composite rubric)", f"Tier {nb6['model_tier']} ({nb6['composite_score']}/9)", "NB6"],
    ["BCBS 239 data-quality gate", f"{dq_pass}/{len(dq['checks'])} checks passed, {dq['n_rows']:,} rows", "NB7"],
    ["Governance sign-off readiness", f"{ALL_PASS}/{ALL_CHECKS} checks Pass, {ALL_TBD} outside pipeline scope", "NB9"],
]
_t = Table(_kpi_rows, colWidths=[2.6*inch, 2.6*inch, 0.8*inch])
_t.setStyle(TableStyle([
    ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#1F3864")),
    ("TEXTCOLOR", (0, 0), (-1, 0), colors.white),
    ("FONTSIZE", (0, 0), (-1, -1), 8.5),
    ("GRID", (0, 0), (-1, -1), 0.5, colors.HexColor("#DCE3EC")),
    ("ROWBACKGROUNDS", (0, 1), (-1, -1), [colors.white, colors.HexColor("#F6F8FB")]),
    ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
]))
story.append(_t)
story.append(Spacer(1, 14))

# ---- SMART decisions by role ----
story.append(Paragraph("SMART Decisions by Role (auto-derived from real flags across NB1-NB9)", _styles["H2c"]))
for rd in ROLE_DECISIONS:
    story.append(Paragraph(f"<b>{rd['role']}</b> -- {rd['decision']}", _styles["Bodyc"]))
    story.append(Paragraph(f"Real basis: {rd['real_basis']}", _styles["Caption"]))
    _shex = _status_hex(rd["status"])
    story.append(Paragraph(
        f"<font color='{_shex}'><b>{rd['status']}</b></font> -- {rd['status_reason']} <i>(Source: {rd['source']})</i>",
        _styles["Caption"]))
    story.append(Spacer(1, 8))
story.append(PageBreak())

# ---- Section: Model Screening & Validation (NB1) ----
story.append(Paragraph("01-02. Model Screening &amp; Champion Validation (NB1)", _styles["H1c"]))
story.append(RLImage(io.BytesIO(CHART_STAGE_A_PNG), width=5.6*inch, height=2.9*inch))
story.append(Paragraph(
    f"Champion: <b>{nb1['champion_name']}</b> (mean CV PR-AUC {nb1['stage_b'][nb1['champion_name']]['mean_pr_auc']:.4f}) "
    f"vs. runner-up <b>{nb1['runner_up_name']}</b> ({nb1['stage_b'][nb1['runner_up_name']]['mean_pr_auc']:.4f}). "
    f"Honest temporal-split PR-AUC is {nb1['temporal_pr_auc']:.4f} -- a real, disclosed divergence from the "
    f"shuffled CV number, since the temporal split trains on the earlier portion and tests on strictly later "
    f"transactions.", _styles["Bodyc"]))
story.append(Spacer(1, 8))
story.append(Paragraph(
    f"External benchmark check: your precision {pct(bc['your_precision'])} vs. reference {pct(bc['reference_precision'])} "
    f"(gap {pct(bc['precision_gap'])}); your recall {pct(bc['your_recall'])} vs. reference {pct(bc['reference_recall'])} "
    f"(gap {pct(bc['recall_gap'])}). investigate_flag=<b>{bc['investigate_flag']}</b>. {bc['note']}", _styles["Bodyc"]))
story.append(PageBreak())

# ---- Section: Financial Impact ----
story.append(Paragraph("03. Cost-Optimal Threshold &amp; Financial Impact (NB1)", _styles["H1c"]))
story.append(RLImage(io.BytesIO(CHART_FINANCIAL_PNG), width=5.6*inch, height=2.9*inch))
_fin_rows = [["Scenario", "Real Total Cost"]] + [[r["Scenario"], eur(r["Total Cost (EUR)"])] for r in nb1["financial_impact"]]
_tf = Table(_fin_rows, colWidths=[3.2*inch, 2.8*inch])
_tf.setStyle(TableStyle([
    ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#1F3864")), ("TEXTCOLOR", (0, 0), (-1, 0), colors.white),
    ("FONTSIZE", (0, 0), (-1, -1), 8.5), ("GRID", (0, 0), (-1, -1), 0.5, colors.HexColor("#DCE3EC")),
]))
story.append(_tf)
story.append(Spacer(1, 8))
story.append(Paragraph(
    f"Cost model (Master Playbook Section 10, real sourced constants): false-negative cost = amount lost x "
    f"{nb5['run_metadata']['fn_cost_per_dollar_lost']} ({nb5['run_metadata']['fn_cost_source']}); false-positive "
    f"cost = flagged amount x {nb5['run_metadata']['fp_cost_multiplier_pct']}% ({nb5['run_metadata']['fp_cost_source']}). "
    f"EUR/USD {EUR_TO_USD} ({nb5['run_metadata']['eur_to_usd_source']}).", _styles["Caption"]))
story.append(PageBreak())

# ---- Section: Governance Gates, Drift & Adversarial Robustness (NB2) ----
story.append(Paragraph("04-05. Governance Gates, Drift &amp; Adversarial Robustness (NB2)", _styles["H1c"]))
story.append(Paragraph(
    f"Gate 1 structural checks: {g1_pass}/{len(g1)} passed (all_passed={nb2['gate1_all_passed']}). "
    f"Gate 2 CV stability: {nb2['gate2_cv_stability_ok']}. Drift monitoring any_alert=<b>{nb2['drift_monitoring']['any_alert']}</b> "
    f"(worst real feature PSI {psi_worst:.4f}; score KS statistic {nb2['drift_monitoring']['score_ks_statistic']:.4f}).",
    _styles["Bodyc"]))
_bnd = nb2["adversarial_robustness"]["boundary_search"]
story.append(Paragraph(
    f"Adversarial boundary search: {_bnd['n_evadable_within_budget']} of {_bnd['n_fraud_cases_tested']} real "
    f"fraud cases evadable within a {pct(_bnd['max_amount_change_pct_budget'],0)} amount-change budget "
    f"(evasion rate {pct(_bnd['evasion_rate_within_budget'])}, median change for evasion "
    f"{pct(_bnd['median_amount_change_pct_for_evasion'],0)}). Evaluated in-sample -- realistic held-out recall "
    f"ceiling is {pct(nb1['real_recall'])}, not the in-sample 100% baseline.", _styles["Bodyc"]))
story.append(PageBreak())

# ---- Section: Deployment (NB4) ----
story.append(Paragraph("06. Deployment &amp; Serving Consistency (NB4)", _styles["H1c"]))
_lat = nb4["latency_sla_ms"]["client_round_trip"]
story.append(Paragraph(
    f"Real local API latency (client round-trip): mean {_lat['mean']:.2f} ms, p50 {_lat['p50']:.2f} ms, "
    f"p95 {_lat['p95']:.2f} ms, p99 {_lat['p99']:.2f} ms. Training/serving consistency: max abs diff "
    f"{nb4['training_serving_consistency']['max_abs_diff']} over {nb4['training_serving_consistency']['n_rows_tested']} "
    f"real rows (passed={nb4['training_serving_consistency']['passed']}). {nb4['latency_sla_ms']['note']}",
    _styles["Bodyc"]))
story.append(PageBreak())

# ---- Section: Stress Testing (NB5) ----
story.append(Paragraph("07. Deepened Stress Testing (NB5)", _styles["H1c"]))
story.append(RLImage(io.BytesIO(CHART_STRESS_PNG), width=5.6*inch, height=2.9*inch))
_ladder_rows = [["Tier", "Volume x", "Fraud-rate x", "Projected Loss"]] + [
    [s["tier_name"], f"{s['ASSUMPTION_scenario_volume_multiplier']:.1f}x", f"{s['ASSUMPTION_scenario_fraud_rate_multiplier']:.1f}x",
     eur(s["projected_fraud_loss_usd_or_eur"])] for s in nb5["scenario_ladder"]]
_ts = Table(_ladder_rows, colWidths=[1.3*inch, 1.1*inch, 1.1*inch, 2.7*inch])
_ts.setStyle(TableStyle([
    ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#1F3864")), ("TEXTCOLOR", (0, 0), (-1, 0), colors.white),
    ("FONTSIZE", (0, 0), (-1, -1), 8.5), ("GRID", (0, 0), (-1, -1), 0.5, colors.HexColor("#DCE3EC")),
]))
story.append(_ts)
story.append(Spacer(1, 8))
_wg = nb5["grid_sweep"]["worst_combination"]
story.append(Paragraph(
    f"Worst of {nb5['grid_sweep']['n_combinations']} real swept combinations: volume x{_wg['volume_multiplier']}, "
    f"fraud-rate x{_wg['fraud_rate_multiplier']} -&gt; projected loss {eur(_wg['projected_loss_eur'])}.",
    _styles["Bodyc"]))
story.append(PageBreak())

# ---- Section: Model Tiering (NB6) ----
story.append(Paragraph("08. Model Tiering Matrix (NB6)", _styles["H1c"]))
_rub = nb6["rubric"]
_tier_rows = [
    ["Dimension", "Real Input", "Score"],
    ["Financial Exposure", f"{_rub['financial_exposure']['stress_multiple']:.1f}x worst-case vs. current cost", f"{_rub['financial_exposure']['score']}/3"],
    ["Regulatory Scrutiny", f"{len(_rub['regulatory_scrutiny']['open_flags'])} real open flags", f"{_rub['regulatory_scrutiny']['score']}/3"],
    ["Customer Impact", f"{_rub['customer_impact']['real_fp']} real FPs / {_rub['customer_impact']['real_n_total']:,}", f"{_rub['customer_impact']['score']}/3"],
]
_tt = Table(_tier_rows, colWidths=[1.8*inch, 3.2*inch, 1*inch])
_tt.setStyle(TableStyle([
    ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#1F3864")), ("TEXTCOLOR", (0, 0), (-1, 0), colors.white),
    ("FONTSIZE", (0, 0), (-1, -1), 8.5), ("GRID", (0, 0), (-1, -1), 0.5, colors.HexColor("#DCE3EC")),
]))
story.append(_tt)
story.append(Spacer(1, 8))
story.append(Paragraph(
    f"Composite score {nb6['composite_score']}/9 -&gt; <b>Tier {nb6['model_tier']} ({nb6['tier_description']})</b>. "
    f"Retroactive finding: NB3's drift-monitoring history used an unjustified default tier="
    f"{nb6['retroactive_finding']['nb3_tier_used_as_default']}; recommend tier={nb6['retroactive_finding']['recommended_tier_going_forward']} "
    f"going forward.", _styles["Bodyc"]))
story.append(PageBreak())

# ---- Section: BCBS 239 (NB7) ----
story.append(Paragraph("09. BCBS 239 Data Governance (NB7)", _styles["H1c"]))
_bcbs_rows = [["Principle Group", "Status", "Evidence"]] + [
    [m["principle_group"], m["status"].replace("_", " "), Paragraph(m["evidence"], _styles["Caption"])]
    for m in nb7["bcbs239_mapping"]]
_tb = Table(_bcbs_rows, colWidths=[1.3*inch, 1.1*inch, 3.6*inch])
_tb.setStyle(TableStyle([
    ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#1F3864")), ("TEXTCOLOR", (0, 0), (-1, 0), colors.white),
    ("FONTSIZE", (0, 0), (-1, -1), 8), ("GRID", (0, 0), (-1, -1), 0.5, colors.HexColor("#DCE3EC")),
    ("VALIGN", (0, 0), (-1, -1), "TOP"),
]))
story.append(_tb)
story.append(PageBreak())

# ---- Section: Regulatory & Oversight (NB8) ----
story.append(Paragraph("10. Regulatory Applicability &amp; Human Oversight (NB8)", _styles["H1c"]))
_reg_rows = [["Framework", "Jurisdiction"]] + [[f["framework"], f["jurisdiction"]] for f in nb8["regulatory_applicability"]]
_tr = Table(_reg_rows, colWidths=[4.2*inch, 1.8*inch])
_tr.setStyle(TableStyle([
    ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#1F3864")), ("TEXTCOLOR", (0, 0), (-1, 0), colors.white),
    ("FONTSIZE", (0, 0), (-1, -1), 8.5), ("GRID", (0, 0), (-1, -1), 0.5, colors.HexColor("#DCE3EC")),
]))
story.append(_tr)
story.append(Spacer(1, 8))
_ho = nb8["human_oversight"]
story.append(Paragraph(
    f"Human-review band (ASSUMPTION +/-{_ho['band_half_width_ASSUMPTION']} around the real threshold): "
    f"{_ho['band_lo']:.4f}-{_ho['band_hi']:.4f}. {_ho['n_in_band']} of {_ho['n_total']:,} real transactions fall "
    f"in-band ({_ho['n_fraud_in_band']} fraud, {_ho['n_legit_in_band']} legit).", _styles["Bodyc"]))
story.append(PageBreak())

# ---- Section: Governance Sign-Off Readiness (NB9) ----
story.append(Paragraph("11. Governance Sign-Off Readiness (NB9)", _styles["H1c"]))
_tier_summary_rows = [
    ["Tier", "Pass", "Conditional", "Outside Scope (TBD)", "Total"],
    ["1 -- Technical Lead", T1["pass"], T1["cond"], T1["tbd"], T1["total"]],
    ["2 -- Model Risk Manager", T2["pass"], T2["cond"], T2["tbd"], T2["total"]],
    ["3 -- CCO", T3["pass"], T3["cond"], T3["tbd"], T3["total"]],
    ["4 -- Business Owner", T4["pass"], T4["cond"], T4["tbd"], T4["total"]],
]
_tg = Table(_tier_summary_rows, colWidths=[1.8*inch, 0.7*inch, 1*inch, 1.3*inch, 0.8*inch])
_tg.setStyle(TableStyle([
    ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#1F3864")), ("TEXTCOLOR", (0, 0), (-1, 0), colors.white),
    ("FONTSIZE", (0, 0), (-1, -1), 8.5), ("GRID", (0, 0), (-1, -1), 0.5, colors.HexColor("#DCE3EC")),
]))
story.append(_tg)
story.append(Spacer(1, 8))
story.append(Paragraph(
    "Every Approve/Conditional/Reject decision and every Name/Date/Signature field in the underlying governance "
    "template is intentionally left blank -- this rollup assembles real evidence, it does not fabricate a human "
    "decision.", _styles["Caption"]))

doc.build(story)
print("Written PDF:", _pdf_path, "| pages: (see file) | bytes:", os.path.getsize(_pdf_path))

##############################################################################
# 2) PPTX -- C-suite presentation deck (python-pptx)
##############################################################################
from pptx import Presentation
from pptx.util import Inches, Pt, Emu
from pptx.dml.color import RGBColor
from pptx.enum.text import PP_ALIGN

INK = RGBColor(0x12, 0x21, 0x3B)
ACCENT = RGBColor(0x1F, 0x38, 0x64)
GOOD = RGBColor(0x1E, 0x7A, 0x4C)
WARN = RGBColor(0xA5, 0x69, 0x0E)
CRIT = RGBColor(0xB0, 0x34, 0x2A)
SOFT = RGBColor(0x5B, 0x6B, 0x85)

def _status_rgb(s):
    return {"Go": GOOD, "Conditional": WARN, "Hold": CRIT}.get(s, SOFT)

prs = Presentation()
prs.slide_width = Inches(13.333)
prs.slide_height = Inches(7.5)
BLANK = prs.slide_layouts[6]

def _add_title(slide, text, sub=None):
    tb = slide.shapes.add_textbox(Inches(0.5), Inches(0.35), Inches(12.3), Inches(1.0))
    tf = tb.text_frame
    tf.text = text
    tf.paragraphs[0].font.size = Pt(30)
    tf.paragraphs[0].font.bold = True
    tf.paragraphs[0].font.color.rgb = ACCENT
    if sub:
        p_ = tf.add_paragraph()
        p_.text = sub
        p_.font.size = Pt(14)
        p_.font.color.rgb = SOFT

def _add_bullets(slide, items, left=0.5, top=1.5, width=6.0, height=5.4, size=13):
    tb = slide.shapes.add_textbox(Inches(left), Inches(top), Inches(width), Inches(height))
    tf = tb.text_frame
    tf.word_wrap = True
    for i, it in enumerate(items):
        p_ = tf.paragraphs[0] if i == 0 else tf.add_paragraph()
        p_.text = "\u2022 " + it
        p_.font.size = Pt(size)
        p_.font.color.rgb = INK
        p_.space_after = Pt(8)

def _add_image(slide, png_bytes, left, top, width):
    slide.shapes.add_picture(io.BytesIO(png_bytes), Inches(left), Inches(top), width=Inches(width))

# Slide 1 -- Title
s = prs.slides.add_slide(BLANK)
tb = s.shapes.add_textbox(Inches(0.8), Inches(2.6), Inches(11.7), Inches(2.2))
tf = tb.text_frame
tf.text = "Fraud Detection Platform"
tf.paragraphs[0].font.size = Pt(44)
tf.paragraphs[0].font.bold = True
tf.paragraphs[0].font.color.rgb = ACCENT
p_ = tf.add_paragraph(); p_.text = "Executive Rollup — Real Results, NB1–NB9"
p_.font.size = Pt(22); p_.font.color.rgb = INK
p_ = tf.add_paragraph()
p_.text = f"Prepared by Nandagopal  |  {datetime.now(timezone.utc).strftime('%Y-%m-%d')}  |  Zero-fabrication policy"
p_.font.size = Pt(13); p_.font.color.rgb = SOFT

# Slide 2 -- Executive KPI summary
s = prs.slides.add_slide(BLANK)
_add_title(s, "Executive Summary", "Every figure below is loaded verbatim from NB1–NB9's real on-disk outputs")
_kpi_items = [
    f"Champion model: {nb1['champion_name']}  (CV PR-AUC {nb1['stage_b'][nb1['champion_name']]['mean_pr_auc']:.4f}, temporal-split {nb1['temporal_pr_auc']:.4f})",
    f"Savings vs. no model: {eur(nb1['savings_vs_no_model'])}  |  vs. naive threshold: {eur(nb1['savings_vs_naive_05'])}",
    f"Model tier: Tier {nb6['model_tier']} — {nb6['tier_description']}  (composite {nb6['composite_score']}/9)",
    f"BCBS 239 data-quality gate: {dq_pass}/{len(dq['checks'])} checks passed on {dq['n_rows']:,} real rows",
    f"Governance sign-off readiness: {ALL_PASS}/{ALL_CHECKS} checks Pass  ({ALL_TBD} outside pipeline scope)",
    f"Real local API latency: p99 {nb4['latency_sla_ms']['client_round_trip']['p99']:.2f} ms",
]
_add_bullets(s, _kpi_items, width=12.0, size=16)

# Slide 3 -- Model Screening & Validation
s = prs.slides.add_slide(BLANK)
_add_title(s, "Model Screening & Validation", "NB1 — 5 real candidates screened, champion selected by 5-fold CV")
_add_image(s, CHART_STAGE_A_PNG, 0.5, 1.6, 6.4)
_add_bullets(s, [
    f"Champion: {nb1['champion_name']} vs. runner-up {nb1['runner_up_name']}",
    f"Temporal PR-AUC {nb1['temporal_pr_auc']:.4f} vs. CV {nb1['stage_b'][nb1['champion_name']]['mean_pr_auc']:.4f} — real, disclosed gap",
    f"Benchmark investigate_flag: {bc['investigate_flag']}  (precision gap {pct(bc['precision_gap'])})",
], left=7.2, top=1.6, width=5.6, size=14)

# Slide 4 -- Financial Impact
s = prs.slides.add_slide(BLANK)
_add_title(s, "Cost-Optimal Threshold & Financial Impact", "NB1 real measured totals")
_add_image(s, CHART_FINANCIAL_PNG, 0.5, 1.6, 6.4)
_add_bullets(s, [f"{r['Scenario']}: {eur(r['Total Cost (EUR)'])}" for r in nb1["financial_impact"]],
             left=7.2, top=1.6, width=5.6, size=14)

# Slide 5 -- Stress Testing
s = prs.slides.add_slide(BLANK)
_add_title(s, "Deepened Stress Testing", "NB5 real 3-tier ladder + 105-combo grid sweep")
_add_image(s, CHART_STRESS_PNG, 0.5, 1.6, 6.4)
_add_bullets(s, [
    f"Worst of {nb5['grid_sweep']['n_combinations']} swept combinations: volume x{_wg['volume_multiplier']}, "
    f"fraud-rate x{_wg['fraud_rate_multiplier']} -> {eur(_wg['projected_loss_eur'])}",
    f"Reverse stress: {nb5['reverse_stress_test']['results'][0]['required_fraud_rate_multiplier']:.2f}x fraud-rate "
    f"needed to double today's real cost",
], left=7.2, top=1.6, width=5.6, size=14)

# Slide 6 -- Governance readiness
s = prs.slides.add_slide(BLANK)
_add_title(s, "Governance Sign-Off Readiness", "NB9 real evidence-populated 4-tier dry-run — no decision fabricated")
_rows_tbl = [("Tier", "Pass", "Conditional", "Outside Scope"),
             ("1 — Technical Lead", str(T1["pass"]) + "/" + str(T1["total"]), str(T1["cond"]), str(T1["tbd"])),
             ("2 — Model Risk Manager", str(T2["pass"]) + "/" + str(T2["total"]), str(T2["cond"]), str(T2["tbd"])),
             ("3 — CCO", str(T3["pass"]) + "/" + str(T3["total"]), str(T3["cond"]), str(T3["tbd"])),
             ("4 — Business Owner", str(T4["pass"]) + "/" + str(T4["total"]), str(T4["cond"]), str(T4["tbd"]))]
_gtbl = s.shapes.add_table(len(_rows_tbl), 4, Inches(0.6), Inches(1.7), Inches(9.5), Inches(2.6)).table
for ci, htext in enumerate(_rows_tbl[0]):
    cell = _gtbl.cell(0, ci); cell.text = htext
    cell.text_frame.paragraphs[0].font.bold = True
    cell.text_frame.paragraphs[0].font.color.rgb = RGBColor(0xFF, 0xFF, 0xFF)
    cell.fill.solid(); cell.fill.fore_color.rgb = ACCENT
for ri in range(1, len(_rows_tbl)):
    for ci, val in enumerate(_rows_tbl[ri]):
        cell = _gtbl.cell(ri, ci); cell.text = val
        cell.text_frame.paragraphs[0].font.size = Pt(13)
_add_bullets(s, [
    "Every Approve/Conditional/Reject checkbox and Name/Date/Signature field is left exactly blank — this "
    "package assembles real evidence, it does not fabricate a human decision.",
], left=0.6, top=4.6, width=11.5, size=13)

# Slides 7..N -- SMART decisions by role (2 per slide)
_per_slide = 2
for _i in range(0, len(ROLE_DECISIONS), _per_slide):
    s = prs.slides.add_slide(BLANK)
    _add_title(s, "SMART Decisions by Role", "Auto-derived from real flags across NB1–NB9")
    _top = 1.6
    for rd in ROLE_DECISIONS[_i:_i + _per_slide]:
        box = s.shapes.add_textbox(Inches(0.6), Inches(_top), Inches(12.1), Inches(2.7))
        tf = box.text_frame
        tf.word_wrap = True
        tf.text = f"{rd['role']} — {rd['decision']}"
        tf.paragraphs[0].font.size = Pt(17); tf.paragraphs[0].font.bold = True; tf.paragraphs[0].font.color.rgb = INK
        p1 = tf.add_paragraph(); p1.text = "Real basis: " + rd["real_basis"]
        p1.font.size = Pt(12); p1.font.color.rgb = SOFT
        p2 = tf.add_paragraph(); p2.text = f"{rd['status']} — {rd['status_reason']}  (Source: {rd['source']})"
        p2.font.size = Pt(12); p2.font.bold = True; p2.font.color.rgb = _status_rgb(rd["status"])
        _top += 2.9

_pptx_path = os.path.join(RESULTS_DIR, "Fraud_Detection_Executive_Rollup_Deck.pptx")
prs.save(_pptx_path)
print("Written PPTX:", _pptx_path, "| slides:", len(prs.slides.__iter__.__self__._sldIdLst), "| bytes:", os.path.getsize(_pptx_path))

##############################################################################
# 3) XLSX -- role-based decision scorecard workbook (openpyxl)
##############################################################################
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

wb = Workbook()

HEADER_FILL = PatternFill("solid", fgColor="1F3864")
HEADER_FONT = Font(color="FFFFFF", bold=True)
GOOD_FILL = PatternFill("solid", fgColor="E3F3EA")
WARN_FILL = PatternFill("solid", fgColor="FBF0DC")
CRIT_FILL = PatternFill("solid", fgColor="FBE7E5")
THIN = Border(*(Side(style="thin", color="DCE3EC"),) * 4)

def _style_header(ws, row, ncols):
    for c in range(1, ncols + 1):
        cell = ws.cell(row=row, column=c)
        cell.fill = HEADER_FILL
        cell.font = HEADER_FONT
        cell.border = THIN
        cell.alignment = Alignment(vertical="center", wrap_text=True)

def _autosize(ws, widths):
    for i, w in enumerate(widths, start=1):
        ws.column_dimensions[get_column_letter(i)].width = w

# ---- Sheet: KPI Summary ----
ws = wb.active
ws.title = "KPI Summary"
ws.append(["Fraud Detection Platform -- Executive KPI Summary (auto-picked from NB1-NB9 real outputs)"])
ws["A1"].font = Font(bold=True, size=14, color="1F3864")
ws.append([])
ws.append(["Metric", "Real Value", "Source Notebook"])
_style_header(ws, 3, 3)
_kv = [
    ("Champion model", nb1["champion_name"], "NB1"),
    ("Champion CV PR-AUC (5-fold)", round(nb1["stage_b"][nb1["champion_name"]]["mean_pr_auc"], 4), "NB1"),
    ("Temporal-split PR-AUC (honest)", round(nb1["temporal_pr_auc"], 4), "NB1"),
    ("Cost-optimal threshold", round(nb1["threshold_result"]["threshold"], 4), "NB1"),
    ("Real precision (OOF)", round(nb1["real_precision"], 4), "NB1"),
    ("Real recall (OOF)", round(nb1["real_recall"], 4), "NB1"),
    ("Total cost -- no model (EUR)", round(nb1["financial_impact"][0]["Total Cost (EUR)"], 2), "NB1"),
    ("Total cost -- naive 0.5 (EUR)", round(nb1["financial_impact"][1]["Total Cost (EUR)"], 2), "NB1"),
    ("Total cost -- cost-optimal (EUR)", round(nb1["financial_impact"][2]["Total Cost (EUR)"], 2), "NB1"),
    ("Savings vs. no model (EUR)", None, "NB1 (live formula)"),
    ("Model tier", nb6["model_tier"], "NB6"),
    ("Model tier composite score (/9)", nb6["composite_score"], "NB6"),
    ("BCBS239 checks passed", f"{dq_pass}/{len(dq['checks'])}", "NB7"),
    ("Governance checks Pass", f"{ALL_PASS}/{ALL_CHECKS}", "NB9"),
    ("Governance checks outside pipeline scope", ALL_TBD, "NB9"),
]
_start_row = 4
for r in _kv:
    ws.append(list(r))
for rr in range(_start_row, _start_row + len(_kv)):
    for cc in range(1, 4):
        ws.cell(row=rr, column=cc).border = THIN
# Real Excel formula: savings vs. no model = no-model cost - optimal cost (live-recalculating if edited)
ws.cell(row=_start_row + 9, column=2, value=f"=B{_start_row+6}-B{_start_row+8}")
_autosize(ws, [34, 20, 22])

# ---- Sheet: Role Decision Matrix ----
ws = wb.create_sheet("Role Decision Matrix")
ws.append(["Role", "Decision", "Real Basis", "Status", "Status Reason", "Source"])
_style_header(ws, 1, 6)
for rd in ROLE_DECISIONS:
    ws.append([rd["role"], rd["decision"], rd["real_basis"], rd["status"], rd["status_reason"], rd["source"]])
for rr in range(2, 2 + len(ROLE_DECISIONS)):
    fill = {"Go": GOOD_FILL, "Conditional": WARN_FILL, "Hold": CRIT_FILL}.get(ws.cell(row=rr, column=4).value, None)
    for cc in range(1, 7):
        cell = ws.cell(row=rr, column=cc)
        cell.border = THIN
        cell.alignment = Alignment(wrap_text=True, vertical="top")
        if cc == 4 and fill:
            cell.fill = fill
            cell.font = Font(bold=True)
_autosize(ws, [22, 34, 55, 12, 45, 14])

# ---- Sheet: NB1 Model Screening & Validation ----
ws = wb.create_sheet("NB1 Screening & Validation")
ws.append(["Rank", "Model", "Validation PR-AUC", "Status"])
_style_header(ws, 1, 4)
for i, d in enumerate(nb1["stage_a"]):
    ws.append([i + 1, d["model"], round(d["val_pr_auc"], 4), "Advances" if i < 2 else "Screened out"])
for rr in range(2, 2 + len(nb1["stage_a"])):
    for cc in range(1, 5):
        ws.cell(row=rr, column=cc).border = THIN
_autosize(ws, [8, 22, 18, 16])

# ---- Sheet: NB2 Governance Gates & Drift ----
ws = wb.create_sheet("NB2 Gates & Drift")
ws.append(["Gate 1 structural check", "Passed"])
_style_header(ws, 1, 2)
for k, v in g1.items():
    ws.append([k, v])
ws.append([])
ws.append(["Feature", "PSI (early vs. late)"])
_style_header(ws, ws.max_row, 2)
for k, v in nb2["drift_monitoring"]["feature_psi"].items():
    ws.append([k, round(v, 4)])
_autosize(ws, [36, 20])

# ---- Sheet: NB4 Deployment ----
ws = wb.create_sheet("NB4 Deployment")
ws.append(["Metric", "Value (ms)"])
_style_header(ws, 1, 2)
for k, v in nb4["latency_sla_ms"]["client_round_trip"].items():
    ws.append([f"client_round_trip.{k}", round(v, 4)])
for k, v in nb4["latency_sla_ms"]["server_reported"].items():
    ws.append([f"server_reported.{k}", round(v, 4)])
_autosize(ws, [30, 16])

# ---- Sheet: NB5 Stress Testing ----
ws = wb.create_sheet("NB5 Stress Testing")
ws.append(["Tier", "Volume x", "Fraud-rate x", "Projected Loss (EUR)"])
_style_header(ws, 1, 4)
for s5 in nb5["scenario_ladder"]:
    ws.append([s5["tier_name"], s5["ASSUMPTION_scenario_volume_multiplier"],
               s5["ASSUMPTION_scenario_fraud_rate_multiplier"], round(s5["projected_fraud_loss_usd_or_eur"], 2)])
ws.append([])
ws.append(["Reverse Stress -- Target Cost Multiple", "Required Fraud-Rate x", "Required Fraud Rate"])
_style_header(ws, ws.max_row, 3)
for r5 in nb5["reverse_stress_test"]["results"]:
    ws.append([r5["target_multiple_of_current_cost"], round(r5["required_fraud_rate_multiplier"], 2),
               round(r5["required_fraud_rate"], 5)])
_autosize(ws, [30, 18, 18, 20])

# ---- Sheet: NB6 Model Tiering ----
ws = wb.create_sheet("NB6 Model Tiering")
ws.append(["Dimension", "Real Input", "Score (/3)"])
_style_header(ws, 1, 3)
ws.append(["Financial Exposure", f"{_rub['financial_exposure']['stress_multiple']:.2f}x worst-case", _rub["financial_exposure"]["score"]])
ws.append(["Regulatory Scrutiny", f"{len(_rub['regulatory_scrutiny']['open_flags'])} open flags", _rub["regulatory_scrutiny"]["score"]])
ws.append(["Customer Impact", f"{_rub['customer_impact']['real_fp']} real FPs", _rub["customer_impact"]["score"]])
ws.append(["Composite", f"Tier {nb6['model_tier']} -- {nb6['tier_description']}", nb6["composite_score"]])
_autosize(ws, [22, 44, 12])

# ---- Sheet: NB7 BCBS239 ----
ws = wb.create_sheet("NB7 BCBS239")
ws.append(["Principle Group", "Status", "Evidence"])
_style_header(ws, 1, 3)
for m in nb7["bcbs239_mapping"]:
    ws.append([m["principle_group"], m["status"], m["evidence"]])
for rr in range(2, 2 + len(nb7["bcbs239_mapping"])):
    ws.cell(row=rr, column=3).alignment = Alignment(wrap_text=True, vertical="top")
_autosize(ws, [20, 20, 70])

# ---- Sheet: NB8 Regulatory & Oversight ----
ws = wb.create_sheet("NB8 Regulatory & Oversight")
ws.append(["Framework", "Jurisdiction", "Relevance"])
_style_header(ws, 1, 3)
for f8 in nb8["regulatory_applicability"]:
    ws.append([f8["framework"], f8["jurisdiction"], f8["relevance"]])
for rr in range(2, 2 + len(nb8["regulatory_applicability"])):
    ws.cell(row=rr, column=3).alignment = Alignment(wrap_text=True, vertical="top")
_autosize(ws, [34, 14, 80])

# ---- Sheet: NB9 Governance Sign-Off ----
ws = wb.create_sheet("NB9 Governance Sign-Off")
ws.append(["Tier", "Check", "Real Result / Finding", "Verdict"])
_style_header(ws, 1, 4)
for _tname, _rows in (("Tier 1 -- Technical Lead", nb9["tier1_rows"]), ("Tier 2 -- Model Risk Manager", nb9["tier2_rows"]),
                       ("Tier 3 -- CCO", nb9["tier3_rows"]), ("Tier 4 -- Business Owner", nb9["tier4_rows"])):
    for row in _rows:
        ws.append([_tname, row[0], row[1], row[2]])
for rr in range(2, ws.max_row + 1):
    verdict = str(ws.cell(row=rr, column=4).value or "")
    fill = GOOD_FILL if verdict.startswith("Pass") else (CRIT_FILL if verdict.startswith("TBD") else WARN_FILL)
    for cc in range(1, 5):
        cell = ws.cell(row=rr, column=cc)
        cell.border = THIN
        cell.alignment = Alignment(wrap_text=True, vertical="top")
        if cc == 4:
            cell.fill = fill
_autosize(ws, [22, 42, 60, 30])

_xlsx_path = os.path.join(RESULTS_DIR, "Fraud_Detection_Executive_Rollup_Scorecard.xlsx")
wb.save(_xlsx_path)
print("Written XLSX:", _xlsx_path, "| sheets:", wb.sheetnames, "| bytes:", os.path.getsize(_xlsx_path))

##############################################################################
# SAVE NOTEBOOK 11 RESULTS
##############################################################################
nb11_report = {
    "run_metadata": {"random_seed": RANDOM_SEED, "n_threads": _N_THREADS,
                      "generated_at_utc": datetime.now(timezone.utc).isoformat()},
    "outputs": {"pdf": _pdf_path, "pptx": _pptx_path, "xlsx": _xlsx_path},
    "role_decisions_count": len(ROLE_DECISIONS),
    "role_decisions_status_counts": {
        s: sum(1 for rd in ROLE_DECISIONS if rd["status"] == s) for s in ("Go", "Conditional", "Hold")
    },
}
with open(os.path.join(RESULTS_DIR, "nb11_report.json"), "w", encoding="utf-8") as f:
    json.dump(nb11_report, f, indent=2, default=str)

_ram_end = psutil.virtual_memory()
_total_elapsed = time.time() - _RUN_T0
print("=" * 70)
print(f"RAM at finish: {_ram_end.percent:.1f}% used ({_ram_end.used/1e9:.2f} GB / {_ram_end.total/1e9:.2f} GB)")
print(f"Total notebook wall-clock time: {_total_elapsed:.2f}s (real, measured).")
print("Notebook 11 complete. Executive rollup written to:", RESULTS_DIR)
